# Marketplace safety

## 1. Dataförståelse & EDA

In [ ]:
import pandas as pd


df = pd.read_csv("data/historical_data.csv")
X_full = df.drop(columns="is_suspicious")
y_full = df["is_suspicious"]



print(f"Antal rader och kolumner: {df.shape} \n") #datasetstorlek
print(f"{df.info()} \n") #datatyper
print(f"Antal misstänkta(1)/icke misstänkta(0):\n{df["is_suspicious"].value_counts()}") #targetfördelning


In [ ]:
print(f"Antal saknade värden: \n{df.isna().sum()}") #enklare överblick över saknade värden

display(df.head())
display(df.describe().T)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


df_for_eda = X_full.copy()
df_for_eda["target"] = y_full


correlation_matrix = df_for_eda.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(12,5))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
plt.title("Heatmap / korrelationsmatris")

### Korrelationsmatris
Här ser vi att target påverkas mest av "contains_off_platfrom", "prev_reports_30d", "account_age_days", "urgency_words".  
Generellt visar det låg korrelation vilket idikerar att alla bidrar med information.

In [ ]:
sns.histplot(df["price"], bins=30)
plt.title("Prisfördelning")
plt.xlabel("Pris")
plt.ylabel("Antal")
plt.show()

Grafen visar att medianen är bäst för imputer strategy, för att hantera saknade värden.

In [ ]:
sns.histplot(df["time_to_first_response_min"], bins=30)
plt.title("Tid för första svar i min")
plt.xlabel("Antal min")
plt.ylabel("Antal")
plt.show()

Grafen visar att medianen är bäst för imputer strategy, för att hantera saknade värden.

## 2. Train/test + preprocessing

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

numeric_features = X_full.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_full.select_dtypes(exclude=["number"]).columns.tolist()


numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

## 3. Modellering och jämförelse

In [ ]:
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier


logreg = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
randforest = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
decisiontree = DecisionTreeClassifier(class_weight="balanced", random_state=42)

pipe_logreg = Pipeline([("preprocess", preprocessor), ("model", logreg)])
pipe_randforest = Pipeline([("preprocess", preprocessor), ("model", randforest)])
pipe_decisiontree = Pipeline([("preprocess", preprocessor), ("model", decisiontree)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = "precision"

baseline_rows = []

for name, pipe in [("LogisticRegression", pipe_logreg), ("RandomForest", pipe_randforest), ("DecisionTree", pipe_decisiontree)]:
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=SCORING)
    baseline_rows.append({"model": name, "mean": scores.mean(), "std": scores.std()})

baseline_table = pd.DataFrame(baseline_rows).sort_values("mean", ascending=False)
baseline_table

## 4. Hyperparameter tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_logreg = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__solver": ["liblinear", "lbfgs"],
}

grid_logreg = GridSearchCV(
    estimator=pipe_logreg,
    param_grid=param_grid_logreg,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1
)
grid_logreg.fit(X_train, y_train)

cv_results = pd.DataFrame(grid_logreg.cv_results_)
best_row = cv_results.loc[grid_logreg.best_index_]

print("Best params:", grid_logreg.best_params_)
print(f"Mean CV {SCORING}:", round(best_row["mean_test_score"], 3))
print(f"Std CV {SCORING}:", round(best_row["std_test_score"], 3))